#Initialization

In [0]:

from pyspark.sql import functions as F
from pyspark.sql.types import StringType, DateType
from pyspark.sql.functions import col, trim
from pyspark.sql.window import Window

**Read Bronze table**

In [0]:
df = spark.table("workspace.bronze.crm_sales_details")

In [0]:
df.describe().display()

In [0]:
df = (
    df
    .withColumn("sls_sales", F.col("sls_sales").cast("DECIMAL"))
    .withColumn("sls_quantity", F.col("sls_quantity").cast("DECIMAL"))
    .withColumn("sls_price", F.col("sls_price").cast("DECIMAL"))
)

In [0]:
df.printSchema()

**Silver Transformations**

Trimming

In [0]:
string_columns = [
    "sls_ord_num",
    "sls_prd_key",
    "sls_cust_id",
    "sls_order_dt",
    "sls_ship_dt",
    "sls_due_dt",
    
]

for column_name in string_columns:
    df = df.withColumn(column_name, F.trim(F.col(column_name)))

#Cleaning Dates

 Convert sales columns only once

**Sales and Price Corrections**

In [0]:
df = (
    df
    .withColumn("sls_sales",    col("sls_sales").cast("INT"))
    .withColumn("sls_quantity", col("sls_quantity").cast("INT"))
    .withColumn("sls_price",    col("sls_price").cast("INT"))
    .withColumn(
        "sls_price",
        F.when(
            col("sls_price").isNull() | (col("sls_price") <= 0),
            F.when(col("sls_quantity") != 0, col("sls_sales") / col("sls_quantity"))
        ).otherwise(col("sls_price"))
    )
)

**Renaming Columns**

In [0]:

RENAME_MAP = {
    "sls_ord_num": "order_number",
    "sls_prd_key": "product_number",
    "sls_cust_id": "customer_id",
    "sls_order_dt": "order_date",
    "sls_ship_dt": "ship_date",
    "sls_due_dt": "due_date",
    "sls_sales": "sales_amount",
    "sls_quantity": "quantity",
    "sls_price": "price"
}
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

In [0]:
df.explain(True)

**Sanity checks of dataframe**

In [0]:
# 1. Columns and types look right?
df.printSchema()

# 2. Preview the data
df.limit(10).display()

# 3. Row count (should match bronze, since nothing was removed)
print(df.count())

# 4. Any negative or missing sales, quantity, price? (should be 0)
df.filter(
    (F.col("sales_amount") <= 0) | F.col("sales_amount").isNull() |
    (F.col("quantity") <= 0)     | F.col("quantity").isNull() |
    (F.col("price") <= 0)        | F.col("price").isNull()
).display()

# 5. Does sales = quantity x price? (should return 0 rows)
df.filter(F.col("sales_amount") != F.col("quantity") * F.col("price")).display()

# 6. Order date after ship or due date? (should return 0 rows)
df.filter(
    (F.col("order_date") > F.col("ship_date")) |
    (F.col("order_date") > F.col("due_date"))
).display()

#Renaming Columns

In [0]:
RENAME_MAP = {
    "prd_id": "product_id",
    "cat_id": "category_id",
    "prd_key": "product_number",
    "prd_nm": "product_name",
    "prd_cost": "product_cost",
    "prd_line": "product_line",
    "prd_start_dt": "start_date",
    "prd_end_dt": "end_date"
}
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

**Sanity checks of dataframe**

In [0]:
df.limit(10).display()

**Writing Silver Table**

In [0]:

df.write.mode("overwrite").format("delta").saveAsTable("workspace.silver.crm_sales")

**Sanity checks of silver table**

In [0]:

%sql
SELECT * FROM workspace.silver.crm_sales LIMIT 10
     